In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date, datetime
from typing import List, Optional
import statistics

In [5]:
@dataclass
class PetProfile:
    """Static information about a pet. Drives breed-aware weighting only."""
    pet_id: str
    name: str
    species: str                      # "dog" | "cat"
    breed: str
    sex: str                       #  "male" | "female"
    post_code: int
    birth_date: date
    is_high_energy: bool      # e.g. Vizsla, Border Collie -> activity weighted higher
    worming_compliant: bool    # provisional, non-clinical care-compliance nudge
    vaccination_current: bool

@dataclass
class CheckInData:
    """
    A single weekly check-in payload.

    Wellbeing scores use a 1-5 scale (1 = very poor, 5 = excellent).
    Acute signals are booleans and feed ONLY the red-flag override layer.
    """
    pet_id: str
    timestamp: datetime

    # --- Core wellbeing vitals (1-5) ---
    appetite_score: int
    energy_score: int
    sleep_quality_score: int
    activity_minutes: int                 # raw activity, compared against baseline

    # --- Toileting / hydration context ---
    toileting_status: str = "normal"      # "normal" | "more_frequent" | "less_frequent"
    water_intake_status: str = "normal"   # "normal" | "increased" | "decreased"

    # --- Acute / red-flag signals (booleans) ---
    blood_in_stool: bool = False
    vomiting: bool = False
    diarrhoea: bool = False
    breathing_difficulty: bool = False
    collapse: bool = False
    seizure: bool = False
    suspected_toxin_ingestion: bool = False
    unable_to_urinate: bool = False
    severe_bleeding: bool = False
    tick_found: bool = False
    pain_or_discomfort_signs: bool = False

    # --- Optional context (NOT direct health-score drivers) ---
    weight_kg: Optional[float] = None   # weight changes ratical maybe a signal
    owner_concern_level: Optional[int] = None   # 1-5 owner subjective worry, advisory only

@dataclass
class BaselineSummary:
    """The pet's own recent normal, computed from history. Section 4.3."""
    n_checkins: int
    has_baseline: bool
    avg_appetite: Optional[float] = None
    avg_energy: Optional[float] = None
    avg_sleep: Optional[float] = None
    avg_activity: Optional[float] = None

    @classmethod
    def from_history(cls, history: List[CheckInData], min_checkins: int = 4) -> "BaselineSummary":
        n = len(history)
        if n == 0:
            return cls(n_checkins=0, has_baseline=False)
        return cls(
            n_checkins=n,
            has_baseline=n >= min_checkins,
            avg_appetite=statistics.mean(h.appetite_score for h in history),
            avg_energy=statistics.mean(h.energy_score for h in history),
            avg_sleep=statistics.mean(h.sleep_quality_score for h in history),
            avg_activity=statistics.mean(h.activity_minutes for h in history),
        )


@dataclass
class RedFlagResult:
    """
    Output of the safety override layer. Replaces the old, unsafe
    `potential_diseases` field with safe, internal-only escalation fields. Section 4.5.
    """
    triggered: bool
    trigger: Optional[str] = None                      # short machine label, e.g. "collapse"
    severity_tier: Optional[str] = None                # "emergency" | "urgent" | "monitor"
    clinical_reason_for_escalation: Optional[str] = None
    recommended_user_pathway: Optional[str] = None
    vet_validation_required: bool = False

@dataclass
class VitalityScoreResult:
    """Unified, product-facing result. Never conta ins a disease name."""
    pet_id: str
    score: Optional[int]               # 0-100, or None while building baseline / on override
    band: str                          # Bright Green | Medium Green | Watch | Action Needed | Building Baseline
    baseline_status: str               # "established" | "building" | "override"
    confidence: str                    # "high" | "moderate" | "low"
    override: Optional[RedFlagResult]  # populated whenever a red flag fired
    drivers: List[str] = field(default_factory=list)
    explanation: str = ""              # plain-English, safe, non-diagnostic
    trend: str = "n/a"



In [6]:
# Tier 1 emergency signs: any one triggers an emergency pathway immediately.
EMERGENCY_SIGNS = {
    "collapse": "Collapse reported",
    "breathing_difficulty": "Breathing difficulty reported",
    "seizure": "Seizure reported",
    "severe_bleeding": "Severe bleeding reported",
    "suspected_toxin_ingestion": "Suspected toxin ingestion reported",
    "unable_to_urinate": "Inability to urinate reported",
}



In [7]:
class RedFlagEngine:
    """
    Layer 2 - the safety net. Three escalation tiers:
      * emergency : seek emergency care now (no reassuring score is shown)
      * urgent    : action needed / urgent vet review pathway
      * monitor   : watch band with a safe explanation
    Acute signs are NEVER reassured away by the wellbeing math.
    """

    def __init__(self, history: Optional[List[CheckInData]] = None):
        # Recent history lets us spot *repeated* signs (e.g. thirst over weeks).
        self.history = history or []

    def evaluate(self, current: CheckInData) -> RedFlagResult:
        # ---------- TIER 1: EMERGENCY ----------
        for field_name, reason in EMERGENCY_SIGNS.items():
            if getattr(current, field_name):
                return RedFlagResult(
                    triggered=True,
                    trigger=field_name,
                    severity_tier="emergency",
                    clinical_reason_for_escalation=reason,
                    recommended_user_pathway=(
                        "Seek emergency veterinary care now. Reported signs may be serious "
                        "and need urgent professional assessment."
                    ),
                    vet_validation_required=True,
                )

        recent = self.history[-3:]   # look back a few check-ins for "repeated" patterns

        low_energy = current.energy_score <= 2

        # Blood in stool, especially alongside lethargy.
        if current.blood_in_stool:
            reason = "Blood in stool reported"
            if low_energy:
                reason += " together with low energy"
            return RedFlagResult(
                True, "blood_in_stool", "urgent", reason,
                "Reported signs may be concerning. Veterinary review is recommended soon.",
                True,
            )

        # Repeated vomiting across recent check-ins.
        if current.vomiting and any(h.vomiting for h in recent):
            return RedFlagResult(
                True, "repeated_vomiting", "urgent",
                "Repeated vomiting across recent check-ins",
                "Repeated vomiting may need veterinary review, especially if it continues.",
                True,
            )

        # Increased thirst across repeated check-ins.
        if current.water_intake_status == "increased" and any(
            h.water_intake_status == "increased" for h in recent
        ):
            return RedFlagResult(
                True, "persistent_increased_thirst", "urgent",
                "Increased thirst across repeated check-ins",
                "Increased thirst may require veterinary review, especially if it persists or "
                "appears with changes in appetite, weight, urination, or energy.",
                True,
            )

        # Persistent appetite loss across repeated check-ins.
        if current.appetite_score <= 2 and any(h.appetite_score <= 2 for h in recent):
            return RedFlagResult(
                True, "persistent_appetite_loss", "urgent",
                "Persistent low appetite across repeated check-ins",
                "Ongoing reduced appetite may need veterinary review if it continues.",
                True,
            )

        # Signs of pain / painful movement.
        if current.pain_or_discomfort_signs:
            return RedFlagResult(
                True, "pain_signs", "urgent",
                "Signs of pain or discomfort reported",
                "Reported signs of pain may need veterinary review.",
                True,
            )
        
        if current.tick_found:
            return RedFlagResult(
                True, "tick_found", "monitor",
                "Tick found",
                "Remove the tick safely and monitor the area. Seek veterinary advice if you "
                "are unsure or your pet seems unwell.",
                True,
            )

        if current.vomiting or current.diarrhoea:
            sign = "Vomiting" if current.vomiting else "Diarrhoea"
            return RedFlagResult(
                True, "single_gi_upset", "monitor",
                f"{sign} reported on a single check-in",
                "Monitor closely and ensure access to water. Seek veterinary advice if it "
                "continues beyond a day or worsens.",
                True,
            )

        # No red flags. Mild single-day dips are handled by the score's Watch band.
        return RedFlagResult(triggered=False)
        


In [8]:
BAND_BRIGHT = 85
BAND_MEDIUM = 70
BAND_WATCH = 50

# How many score points a full deviation can remove (tuning constant, provisional).
DEVIATION_WEIGHT = 60


class VitalityScoreEngine:
    """
    Layer 1 - the 0-100 wellness trend score.

    Compares the current check-in against the pet's OWN recent baseline rather
    than using fixed deductions. Pure wellbeing math: acute red flags are handled
    separately by RedFlagEngine and must not be folded in here.
    """

    MIN_BASELINE = 4

    def __init__(self, pet: PetProfile, history: List[CheckInData]):
        self.pet = pet
        self.history = history

    def calculate(self, current: CheckInData) -> VitalityScoreResult:
        baseline = BaselineSummary.from_history(self.history, self.MIN_BASELINE)

        # Not enough history -> honest "building baseline" state (section 4.3).
        if not baseline.has_baseline:
            return VitalityScoreResult(
                pet_id=self.pet.pet_id,
                score=None,
                band="Building Baseline",
                baseline_status="building",
                confidence="low",
                override=None,
                drivers=[f"only {baseline.n_checkins} of {self.MIN_BASELINE} check-ins collected"],
                explanation=(
                    f"{self.pet.name} is still building a baseline. With fewer than "
                    f"{self.MIN_BASELINE} recent check-ins we cannot yet tell what is normal "
                    f"for them, so no score is shown. Keep checking in each week."
                ),
                trend="n/a",
            )

        # Negative deviations from baseline (only drops reduce wellbeing).
        drop_appetite = max(0.0, baseline.avg_appetite - current.appetite_score)
        drop_energy = max(0.0, baseline.avg_energy - current.energy_score)
        drop_sleep = max(0.0, baseline.avg_sleep - current.sleep_quality_score)
        drop_activity_frac = 0.0
        if baseline.avg_activity and baseline.avg_activity > 0:
            drop_activity_frac = max(
                0.0, (baseline.avg_activity - current.activity_minutes) / baseline.avg_activity
            )

        # Breed-aware weights (each set sums to 1.0). High-energy breeds weight activity.
        if self.pet.is_high_energy:
            w_app, w_energy, w_sleep, w_act = 0.20, 0.25, 0.10, 0.45
        else:
            w_app, w_energy, w_sleep, w_act = 0.30, 0.30, 0.20, 0.20

        score = 100.0
        drivers: List[str] = []

        score -= drop_appetite * w_app * DEVIATION_WEIGHT
        score -= drop_energy * w_energy * DEVIATION_WEIGHT
        score -= drop_sleep * w_sleep * DEVIATION_WEIGHT
        score -= drop_activity_frac * w_act * DEVIATION_WEIGHT

        if drop_appetite >= 1:
            drivers.append("appetite below usual")
        if drop_energy >= 1:
            drivers.append("energy below usual")
        if drop_sleep >= 1:
            drivers.append("sleep quality below usual")
        if drop_activity_frac >= 0.25:
            drivers.append("activity below usual")

        # Signal vs noise: multiple indicators declining together is more concerning
        # than a single one-day dip (section 4.3 - "multiple indicators declining together").
        declining = sum([
            drop_appetite >= 1,
            drop_energy >= 1,
            drop_sleep >= 1,
            drop_activity_frac >= 0.25,
        ])
        if declining >= 2:
            score -= 12.0
            drivers.append("several indicators declining together")

        # Provisional, non-clinical care-compliance nudge.
        if not self.pet.worming_compliant:
            score -= 8.0
            drivers.append("worming treatment overdue")

        final = int(max(0, min(100, round(score))))
        band = self._band(final)
        confidence = "high" if baseline.n_checkins >= 6 else "moderate"
        if not drivers:
            drivers = ["all tracked indicators in line with usual baseline"]
        trend = "stable" if final >= BAND_MEDIUM else "declining vs baseline"

        return VitalityScoreResult(
            pet_id=self.pet.pet_id,
            score=final,
            band=band,
            baseline_status="established",
            confidence=confidence,
            override=None,
            drivers=drivers,
            explanation=self._explain(band, drivers),
            trend=trend,
        )

    @staticmethod
    def _band(score: int) -> str:
        if score >= BAND_BRIGHT:
            return "Bright Green"
        if score >= BAND_MEDIUM:
            return "Medium Green"
        if score >= BAND_WATCH:
            return "Watch"
        return "Action Needed"

    def _explain(self, band: str, drivers: List[str]) -> str:
        name = self.pet.name
        if band in ("Bright Green", "Medium Green"):
            return (
                f"{name}'s indicators are tracking in line with their usual baseline this week. "
                f"No emergency red flags were reported. Keep up the regular check-ins."
            )
        driver_text = ", ".join(drivers[:2]) if drivers else "small changes from baseline"
        return (
            f"{name}'s score is lower this week mainly because {driver_text} compared with their "
            f"usual baseline. This is a wellbeing trend, not a diagnosis. If it continues or "
            f"worsens, veterinary advice should be considered."
        )


def _override_explanation(pet: PetProfile, rf: RedFlagResult) -> str:
    if rf.severity_tier == "emergency":
        return (
            f"{rf.clinical_reason_for_escalation} for {pet.name}. {rf.recommended_user_pathway} "
            f"This is a safety alert based on reported signs, not a diagnosis."
        )
    return (
        f"{rf.clinical_reason_for_escalation} for {pet.name}. {rf.recommended_user_pathway} "
        f"This is a safety prompt based on reported signs, not a diagnosis."
    )


def process_checkin(
    pet: PetProfile, history: List[CheckInData], current: CheckInData
) -> VitalityScoreResult:
    """
    Full pipeline:
      1. The red-flag override layer runs FIRST (safety net).
      2. An EMERGENCY or URGENT flag replaces the reassuring score with a safety pathway.
      3. Otherwise the 0-100 baseline score is calculated; a MONITOR flag is attached
         as advisory context without hiding the score.
    """
    rf = RedFlagEngine(history=history).evaluate(current)

    if rf.triggered and rf.severity_tier in ("emergency", "urgent"):
        return VitalityScoreResult(
            pet_id=pet.pet_id,
            score=None,                       # score withheld; safety pathway takes over
            band="Action Needed",
            baseline_status="override",
            confidence="high",
            override=rf,
            drivers=[rf.clinical_reason_for_escalation],
            explanation=_override_explanation(pet, rf),
            trend="override",
        )

    result = VitalityScoreEngine(pet, history).calculate(current)

    # Attach a monitor-tier advisory if present (does not hide the wellbeing score).
    if rf.triggered and rf.severity_tier == "monitor":
        result.override = rf
        result.drivers = [rf.clinical_reason_for_escalation] + result.drivers
        result.explanation = f"{result.explanation} {rf.recommended_user_pathway}"

    return result


In [ ]:
import json
from pathlib import Path

TEST_DIR = Path("test_case")

def load_pet_profile(p: dict) -> PetProfile:
    return PetProfile(
        pet_id=p["pet_id"],
        name=p["pet_name"],                    # JSON uses pet_name
        species=p["species"],
        breed=p["breed"],
        sex=p["sex"],
        post_code=int(p["postcode"]),          # JSON uses postcode
        birth_date=date.fromisoformat(p["birth_date"]),
        is_high_energy=p["is_high_energy"],
        worming_compliant=p["worming_compliant"],
        vaccination_current=p["vaccination_current"],
    )

def load_checkin(c: dict) -> CheckInData:
    return CheckInData(
        pet_id=c["pet_id"],
        timestamp=datetime.fromisoformat(c["timestamp"].replace("Z", "+00:00")),
        # appetite_scorew is a typo in the JSON files; fall back gracefully
        appetite_score=c.get("appetite_score", c.get("appetite_scorew")),
        energy_score=c["energy_score"],
        sleep_quality_score=c["sleep_quality_score"],
        activity_minutes=c["activity_minutes"],
        toileting_status=c.get("toileting_status", "normal"),
        water_intake_status=c.get("water_intake_status", "normal"),
        blood_in_stool=c.get("blood_in_stool", False),
        vomiting=c.get("vomiting", False),
        # dỉarhea (non-ASCII) is a typo in the JSON files
        diarrhoea=c.get("diarrhoea", c.get("diarrhea", False)),
        breathing_difficulty=c.get("breathing_difficulty", False),
        collapse=c.get("collapse", False),
        seizure=c.get("seizure", False),
        suspected_toxin_ingestion=c.get("suspected_toxin_ingestion", False),
        unable_to_urinate=c.get("unable_to_urinate", False),
        severe_bleeding=c.get("severe_bleeding", False),
        tick_found=c.get("tick_found", False),
        pain_or_discomfort_signs=c.get("pain_or_discomfort_signs", False),
        weight_kg=c.get("weight_kg"),
        owner_concern_level=c.get("owner_concern_level"),
    )

def make_history(pet_id: str, n: int, appetite: int, energy: int, sleep: int, activity: int) -> List[CheckInData]:
    """Synthetic stable baseline — same scores repeated n times."""
    return [
        CheckInData(
            pet_id=pet_id,
            timestamp=datetime(2026, 1, i + 1),
            appetite_score=appetite,
            energy_score=energy,
            sleep_quality_score=sleep,
            activity_minutes=activity,
        )
        for i in range(n)
    ]


def run_test(label: str, json_path: Path, history: List[CheckInData]) -> None:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    pet = load_pet_profile(data["pet_profile"])
    current = load_checkin(data["check_in_data"])
    result = process_checkin(pet, history, current)

    print(f"{'─' * 62}")
    print(f"  {label}  |  {pet.name} ({pet.breed})")
    print(f"{'─' * 62}")
    print(f"  Score           : {result.score if result.score is not None else 'N/A: override active'}")
    print(f"  Band            : {result.band}")
    print(f"  Baseline status : {result.baseline_status}")
    print(f"  Confidence      : {result.confidence}")
    print(f"  Trend           : {result.trend}")
    print(f"  Drivers         : {', '.join(result.drivers)}")
    if result.override:
        rf = result.override
        print(f"  Red Flag        : [{rf.severity_tier.upper()}] {rf.trigger}")
        print(f"  Pathway         : {rf.recommended_user_pathway}")
    print(f"  Explanation     : {result.explanation}")
    print()

# Test 1: Bonnie: Vizsla, high-energy, clean healthy check-in 
history_1 = make_history("pet_100495", n=4, appetite=4, energy=4, sleep=4, activity=65)
run_test("TEST 1", TEST_DIR / "test_1.json", history_1)

# Test 2: Minh: Scottish Fold, diarrhoea + worming overdue 
history_2 = make_history("pet_100000", n=4, appetite=3, energy=3, sleep=3, activity=40)
run_test("TEST 2", TEST_DIR / "test_2.json", history_2)

# Test 3: Bonnie same profile as test_1 
history_3 = make_history("pet_100495", n=4, appetite=4, energy=4, sleep=4, activity=65)
run_test("TEST 3", TEST_DIR / "test_3.json", history_3)

──────────────────────────────────────────────────────────────
  TEST 1  |  Bonnie (Vizsla)
──────────────────────────────────────────────────────────────
  Score           : 100
  Band            : Bright Green
  Baseline status : established
  Confidence      : moderate
  Trend           : stable
  Drivers         : all tracked indicators in line with usual baseline
  Explanation     : Bonnie's indicators are tracking in line with their usual baseline this week. No emergency red flags were reported. Keep up the regular check-ins.

──────────────────────────────────────────────────────────────
  TEST 2  |  Minh (Scottish Fold)
──────────────────────────────────────────────────────────────
  Score           : 92
  Band            : Bright Green
  Baseline status : established
  Confidence      : moderate
  Trend           : stable
  Drivers         : worming treatment overdue
  Explanation     : Minh's indicators are tracking in line with their usual baseline this week. No emergency re